In [1]:
import os
print(os.listdir('.'))

['.config', 'sample_data']


In [2]:
import pandas as pd

df = pd.read_csv('funnel_events_sample.csv')
print(df.head())
print(df.info())
print(df['step'].unique())

  user_id            step            timestamp
0    U073  email_verified  2026-07-20 04:42:00
1    U003  details_filled  2026-07-20 04:28:00
2    U001  details_filled  2026-07-20 04:14:00
3    U198  details_filled  2026-07-20 04:24:00
4    U129    visited_site  2026-07-20 03:48:00
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 542 entries, 0 to 541
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   user_id    542 non-null    object
 1   step       542 non-null    object
 2   timestamp  542 non-null    object
dtypes: object(3)
memory usage: 12.8+ KB
None
['email_verified' 'details_filled' 'visited_site' 'signup_started'
 'purchase_completed']


In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('funnel_events_sample.csv')

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Define fixed step order
step_order = [
    'visited_site',
    'signup_started',
    'details_filled',
    'email_verified',
    'purchase_completed'
]

# Sort steps according to funnel sequence
df['step_rank'] = df['step'].map(lambda x: step_order.index(x) if x in step_order else np.nan)

# Deduplicate events per user/step (keep earliest timestamp)
df_clean = df.sort_values(['user_id', 'step_rank', 'timestamp']).drop_duplicates(subset=['user_id', 'step'], keep='first')

# Calculate unique users per step
step_counts = df_clean.groupby('step')['user_id'].nunique().reindex(step_order).fillna(0).astype(int)

# Build Funnel Summary Table
funnel_df = pd.DataFrame({'unique_users': step_counts})
funnel_df['pct_of_top'] = (funnel_df['unique_users'] / funnel_df['unique_users'].iloc[0] * 100).round(2)
funnel_df['conversion_rate'] = (funnel_df['unique_users'] / funnel_df['unique_users'].shift(1) * 100).fillna(100.0).round(2)
funnel_df['drop_off_count'] = funnel_df['unique_users'].shift(1) - funnel_df['unique_users']
funnel_df['drop_off_rate'] = (100 - funnel_df['conversion_rate']).round(2)
funnel_df.loc[step_order[0], 'drop_off_count'] = 0
funnel_df.loc[step_order[0], 'drop_off_rate'] = 0.0

print(funnel_df)

# Automate identifying the biggest drop-off stage
# Drop-off can be measured by volume (user loss) or percentage loss from previous step.
biggest_drop_pct_stage = funnel_df['drop_off_rate'].idxmax()
biggest_drop_vol_stage = funnel_df['drop_off_count'].idxmax()

print("\n--- Automated Flagging ---")
print(f"Biggest Drop-off Stage (by % loss): {biggest_drop_pct_stage} ({funnel_df.loc[biggest_drop_pct_stage, 'drop_off_rate']}% dropped off)")
print(f"Biggest Drop-off Stage (by user volume): {biggest_drop_vol_stage} ({funnel_df.loc[biggest_drop_vol_stage, 'drop_off_count']} users lost)")

# Time to convert between consecutive stages
# Pivot table for first timestamp per user per step
pivoted = df_clean.pivot(index='user_id', columns='step', values='timestamp')
pivoted = pivoted[step_order]

time_diffs = {}
for i in range(len(step_order) - 1):
    s1, s2 = step_order[i], step_order[i+1]
    # filter users who reached both s1 and s2 and s2 timestamp >= s1 timestamp
    valid = pivoted[pivoted[s1].notna() & pivoted[s2].notna() & (pivoted[s2] >= pivoted[s1])]
    diff = (valid[s2] - valid[s1]).dt.total_seconds() / 60.0 # in minutes
    time_diffs[f"{s1} -> {s2}"] = diff.mean()

print("\n--- Average Time-to-Convert (Minutes) ---")
for k, v in time_diffs.items():
    print(f"{k}: {v:.2f} mins")

# Segment comparison (e.g. user_id pattern: user ID number even vs odd, or first digit)
# Let's inspect user_ids to see segment opportunities
df['segment'] = df['user_id'].apply(lambda x: 'Segment A (U000-U099)' if int(x.replace('U','')) < 100 else 'Segment B (U100+)')

seg_summary = df.groupby(['segment', 'step'])['user_id'].nunique().unstack()[step_order]
seg_conv = seg_summary.pct_change(axis=1, fill_value=None) # calculate relative or pct of previous
print("\n--- Segment Counts ---")
print(seg_summary)

                    unique_users  pct_of_top  conversion_rate  drop_off_count  \
step                                                                            
visited_site                 200       100.0           100.00             0.0   
signup_started               150        75.0            75.00            50.0   
details_filled                96        48.0            64.00            54.0   
email_verified                52        26.0            54.17            44.0   
purchase_completed            44        22.0            84.62             8.0   

                    drop_off_rate  
step                               
visited_site                 0.00  
signup_started              25.00  
details_filled              36.00  
email_verified              45.83  
purchase_completed          15.38  

--- Automated Flagging ---
Biggest Drop-off Stage (by % loss): email_verified (45.83% dropped off)
Biggest Drop-off Stage (by user volume): details_filled (54.0 users lost)

--- A

/tmp/ipykernel_1785/1179339895.py:70: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  seg_conv = seg_summary.pct_change(axis=1, fill_value=None) # calculate relative or pct of previous


In [4]:
# Let's inspect pivoted dataframe timestamps
pivoted = df_clean.pivot(index='user_id', columns='step', values='timestamp')[step_order]
print(pivoted.head(10))

for i in range(len(step_order) - 1):
    s1, s2 = step_order[i], step_order[i+1]
    valid = pivoted[pivoted[s1].notna() & pivoted[s2].notna()]
    diff = (valid[s2] - valid[s1]).dt.total_seconds() / 60.0
    print(f"{s1} -> {s2}: {len(valid)} users, mean diff = {diff.mean():.2f} mins")

step           visited_site      signup_started      details_filled  \
user_id                                                               
U001    2026-07-20 04:05:00 2026-07-20 04:17:00 2026-07-20 04:14:00   
U002    2026-07-20 04:01:00 2026-07-20 04:08:00 2026-07-20 04:07:00   
U003    2026-07-20 03:58:00 2026-07-20 04:10:00 2026-07-20 04:28:00   
U004    2026-07-20 03:30:00 2026-07-20 04:01:00 2026-07-20 04:09:00   
U005    2026-07-20 03:31:00 2026-07-20 04:17:00 2026-07-20 04:21:00   
U006    2026-07-20 03:40:00 2026-07-20 04:04:00 2026-07-20 04:30:00   
U007    2026-07-20 03:39:00 2026-07-20 04:04:00 2026-07-20 04:09:00   
U008    2026-07-20 03:54:00 2026-07-20 04:03:00 2026-07-20 04:31:00   
U009    2026-07-20 03:45:00 2026-07-20 04:14:00 2026-07-20 04:34:00   
U010    2026-07-20 03:43:00 2026-07-20 04:18:00 2026-07-20 04:31:00   

step    email_verified purchase_completed  
user_id                                    
U001               NaT                NaT  
U002           

In [5]:
# Check steps by user ID ranges or patterns
print(df.groupby(['user_id', 'step']).size().unstack().fillna(0).head(20))
# Let's see which users reach email_verified vs purchase_completed
print("Users reaching email_verified but not purchase_completed:", len(set(df[df['step']=='email_verified']['user_id']) - set(df[df['step']=='purchase_completed']['user_id'])))
print("Users reaching purchase_completed without email_verified:", len(set(df[df['step']=='purchase_completed']['user_id']) - set(df[df['step']=='email_verified']['user_id'])))

step     details_filled  email_verified  purchase_completed  signup_started  \
user_id                                                                       
U001                1.0             0.0                 0.0             1.0   
U002                1.0             0.0                 0.0             1.0   
U003                1.0             0.0                 0.0             1.0   
U004                1.0             0.0                 0.0             1.0   
U005                1.0             0.0                 0.0             1.0   
U006                1.0             0.0                 0.0             1.0   
U007                1.0             0.0                 0.0             1.0   
U008                1.0             0.0                 0.0             1.0   
U009                1.0             0.0                 0.0             1.0   
U010                1.0             0.0                 0.0             1.0   
U011                1.0             0.0             

In [6]:
# Check time difference between details_filled -> purchase_completed for users who completed purchase
valid_pur = pivoted[pivoted['details_filled'].notna() & pivoted['purchase_completed'].notna()]
diff_pur = (valid_pur['purchase_completed'] - valid_pur['details_filled']).dt.total_seconds() / 60.0
print("details_filled -> purchase_completed mean time:", diff_pur.mean(), "mins (N =", len(valid_pur), "users)")

# Check overall step counts
print("\nUnique users per step:")
for step in step_order:
    print(f"{step}: {df[df['step'] == step]['user_id'].nunique()}")

details_filled -> purchase_completed mean time: nan mins (N = 0 users)

Unique users per step:
visited_site: 200
signup_started: 150
details_filled: 96
email_verified: 52
purchase_completed: 44


In [7]:
pur_users = df[df['step'] == 'purchase_completed']['user_id'].unique()
print(pivoted.loc[pur_users].head(10))

step           visited_site      signup_started details_filled email_verified  \
user_id                                                                         
U111    2026-07-20 03:46:00 2026-07-20 03:49:00            NaT            NaT   
U108    2026-07-20 03:41:00 2026-07-20 03:48:00            NaT            NaT   
U118    2026-07-20 03:57:00 2026-07-20 04:11:00            NaT            NaT   
U115    2026-07-20 04:02:00 2026-07-20 03:58:00            NaT            NaT   
U121    2026-07-20 04:06:00 2026-07-20 04:22:00            NaT            NaT   
U133    2026-07-20 04:09:00 2026-07-20 04:16:00            NaT            NaT   
U101    2026-07-20 03:33:00 2026-07-20 03:49:00            NaT            NaT   
U131    2026-07-20 03:34:00 2026-07-20 04:23:00            NaT            NaT   
U138    2026-07-20 03:32:00 2026-07-20 04:22:00            NaT            NaT   
U126    2026-07-20 03:32:00 2026-07-20 03:47:00            NaT            NaT   

step     purchase_completed

In [8]:
# Detailed breakdown for report
total_users = df['user_id'].nunique()
print(f"Total Unique Users in Dataset: {total_users}")

# 1. Basic Step Unique Counts & Funnel Table
step_counts = df.groupby('step')['user_id'].nunique().reindex(step_order)

funnel = pd.DataFrame({
    'Stage': step_order,
    'Unique Users': step_counts.values
})
funnel['% of Top (Overall % Goal)'] = (funnel['Unique Users'] / funnel['Unique Users'].iloc[0] * 100).round(2)
funnel['Stage-to-Stage Conversion Rate (%)'] = (funnel['Unique Users'] / funnel['Unique Users'].shift(1) * 100).fillna(100.0).round(2)
funnel['Users Dropped Off'] = (funnel['Unique Users'].shift(1) - funnel['Unique Users']).fillna(0).astype(int)
funnel['Drop-off Rate (%)'] = (100.0 - funnel['Stage-to-Stage Conversion Rate (%)']).round(2)
funnel.loc[0, 'Drop-off Rate (%)'] = 0.0

print("\n--- Summary Table ---")
print(funnel.to_string(index=False))

# Identify biggest drop-off stage
max_drop_pct_idx = funnel['Drop-off Rate (%)'].idxmax()
max_drop_vol_idx = funnel['Users Dropped Off'].idxmax()

print("\n--- Drop-off Analysis ---")
print(f"Largest Drop-off by %: {funnel.loc[max_drop_pct_idx, 'Stage']} ({funnel.loc[max_drop_pct_idx, 'Drop-off Rate (%)']}% loss from previous stage)")
print(f"Largest Drop-off by Volume: {funnel.loc[max_drop_vol_idx, 'Stage']} ({funnel.loc[max_drop_vol_idx, 'Users Dropped Off']} users lost)")

# Time to convert between available consecutive pairs
t_visited_signup = (pivoted['signup_started'] - pivoted['visited_site']).dropna().dt.total_seconds() / 60.0
t_signup_details = (pivoted['details_filled'] - pivoted['signup_started']).dropna().dt.total_seconds() / 60.0
t_signup_purchase = (pivoted['purchase_completed'] - pivoted['signup_started']).dropna().dt.total_seconds() / 60.0

print("\n--- Time-to-Convert Metrics ---")
print(f"Visited Site -> Signup Started: {t_visited_signup.mean():.2f} mins (Median: {t_visited_signup.median():.2f} mins)")
print(f"Signup Started -> Details Filled: {t_signup_details.mean():.2f} mins (Median: {t_signup_details.median():.2f} mins)")
print(f"Signup Started -> Purchase Completed (Direct buyers): {t_signup_purchase.mean():.2f} mins (Median: {t_signup_purchase.median():.2f} mins)")

Total Unique Users in Dataset: 200

--- Summary Table ---
             Stage  Unique Users  % of Top (Overall % Goal)  Stage-to-Stage Conversion Rate (%)  Users Dropped Off  Drop-off Rate (%)
      visited_site           200                      100.0                              100.00                  0               0.00
    signup_started           150                       75.0                               75.00                 50              25.00
    details_filled            96                       48.0                               64.00                 54              36.00
    email_verified            52                       26.0                               54.17                 44              45.83
purchase_completed            44                       22.0                               84.62                  8              15.38

--- Drop-off Analysis ---
Largest Drop-off by %: email_verified (45.83% loss from previous stage)
Largest Drop-off by Volume: details_fil

In [9]:
df['user_num'] = df['user_id'].str.replace('U','').astype(int)
df['cohort'] = np.where(df['user_num'] <= 100, 'Cohort 1 (U001-U100)', 'Cohort 2 (U101-U200)')

cohort_summary = df.groupby(['cohort', 'step'])['user_id'].nunique().unstack()[step_order].fillna(0).astype(int)
print("--- Cohort Breakdown ---")
print(cohort_summary)

--- Cohort Breakdown ---
step                  visited_site  signup_started  details_filled  \
cohort                                                               
Cohort 1 (U001-U100)           100             100              46   
Cohort 2 (U101-U200)           100              50              50   

step                  email_verified  purchase_completed  
cohort                                                    
Cohort 1 (U001-U100)              52                   2  
Cohort 2 (U101-U200)               0                  42  


In [10]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))

bars = ax.bar(funnel['Stage'], funnel['Unique Users'], color=['#2b5c8f', '#3670a3', '#4185b8', '#5c9ecc', '#80b8e0'])

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 3, f"{int(yval)}", ha='center', va='bottom', fontweight='bold')

ax.set_title('User Funnel Analysis - Unique Users per Stage', fontsize=12, fontweight='bold')
ax.set_ylabel('Unique Users')
ax.set_ylim(0, 230)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('funnel_chart.png', dpi=300)
plt.close()

In [11]:
# Logic used to detect peak drop-off stages
biggest_drop_pct_stage = funnel_df['drop_off_rate'].idxmax()  # -> 'email_verified' (45.83%)
biggest_drop_vol_stage = funnel_df['drop_off_count'].idxmax()  # -> 'details_filled' (54 users)